Quick notebook to combine the most important parameters for this study into dataframe and save a netcdf

In [1]:
import pandas as pd
import geopandas as gpd
import xarray as xr
import rasterio
from affine import Affine
import numpy as np

In [2]:
temp = gpd.read_file("../Data/Crestlines/SKL_rect_masked.shp").set_index("index_poly")
temp["circ_area"] = gpd.read_file("../Data/Crestlines/SKL_circ.shp").set_index("index_poly").to_crs(3577).area
temp["spacing_median"] = gpd.read_file("../Data/Crestlines/Spacing_3577_from_sects_100.shp").set_index("index_poly").spc_med
temp = temp.join(pd.read_csv("../Data/Crestlines/vol_slope_height_width_si100_so30.csv").set_index("index_poly"))
temp["I_count"] = gpd.sjoin(gpd.read_file("../Data/Crestlines/Interaction_point_sects_100.shp").set_index("index_poly"),
                            gpd.read_file("../Data/Crestlines/SKL_circ.shp").set_index("index_poly"),
                            how="inner",predicate="intersects",lsuffix="left",rsuffix="right").index.value_counts()
temp["I_count"] = temp.I_count.fillna(0)
temp["I_Day"] = temp.I_count/temp.circ_area
temp["I_Marvin"] = temp.I_Day*temp.spacing_median**2
temp[["avg100sig","count_100","100_bdir","linear_std"]] = gpd.read_file("../Data/Crestlines/signed_average_angle.shp").set_index("index_poly")[["avg100sig","count_100","100_bdir","linear_std"]]
temp["daspect"] = np.tan(temp.mean_slope_median*np.pi/180)
temp.drop("geometry",axis=1).to_xarray().to_netcdf("../Data/Netcdfs/aus_dune_topo.nc")